## Using JSON files and Dataframe from 01_data_collection notebook

In [31]:
import json
import pandas as pd

# Loading json files
with open("../data/raw/NCD_BMI_30C.json") as f:
    obesity_adult = json.load(f)["value"]

with open("../data/raw/NCD_BMI_PLUS2C.json") as f:
    obesity_child = json.load(f)["value"]

with open("../data/raw/NCD_BMI_18C.json") as f:
    malnutrition_adult = json.load(f)["value"]

with open("../data/raw/NCD_BMI_MINUS2C.json") as f:
    malnutrition_child = json.load(f)["value"]

df_obesity_adult = pd.DataFrame(obesity_adult)
df_obesity_child = pd.DataFrame(obesity_child)
df_malnutrition_adult = pd.DataFrame(malnutrition_adult)
df_malnutrition_child = pd.DataFrame(malnutrition_child)

#creating age_group and category columns in dataframes
df_obesity_adult["age_group"] = "Adult"
df_obesity_child["age_group"] = "Child/Adolescent"
df_malnutrition_adult["age_group"] = "Adult"
df_malnutrition_child["age_group"] = "Child/Adolescent"

df_obesity_adult["_category"] = "obesity"
df_obesity_child["_category"] = "obesity"
df_malnutrition_adult["_category"] = "malnutrition"
df_malnutrition_child["_category"] = "malnutrition"

#Recreating 2 combined dataframes
df_obesity = pd.concat(
    [df_obesity_adult, df_obesity_child],
    ignore_index=True
)

df_malnutrition = pd.concat(
    [df_malnutrition_adult, df_malnutrition_child],
    ignore_index=True
)
df_obesity = df_obesity[df_obesity["TimeDim"].between(2012,2022)]
df_malnutrition = df_malnutrition[df_malnutrition["TimeDim"].between(2012,2022)]

In [33]:
import pandas as pd
import pycountry

## Column selcetion and renaming

In [35]:
COLS_KEEP=["ParentLocation","Dim1","TimeDim","Low","High","NumericValue","SpatialDim","age_group"]
RENAME = {
    "TimeDim":      "Year",
    "Dim1":         "Gender",
    "NumericValue": "Mean_Estimate",
    "Low":          "LowerBound",
    "High":         "UpperBound",
    "ParentLocation":"Region",
    "SpatialDim":   "Country",
}

## Pycountry converter

In [36]:
SPECIAL_CASES = {
    "GLOBAL":"Global",
    "WB_LMI":"Low & Middle Income",
    "WB_HI":"High Income",
    "WB_LI":"Low Income",
    "EMR":"Eastern Mediterranean Region",
    "EUR":"Europe",
    "AFR":"Africa",
    "SEAR":"South-East Asia Region",
    "WPR":"Western Pacific Region",
    "AMR":"Americas Region",
    "WB_UMI":"Upper Middle Income",
}

def code_to_country(code):
    if pd.isna(code): return None
    code = str(code).strip().upper()
    if code in SPECIAL_CASES:
        return SPECIAL_CASES[code]
    try:
        return pycountry.countries.get(alpha_3=code).name
    except AttributeError:
        return code   # keep original if not found

In [37]:
def clean_df(df, label_col, level_fn):
    df = df[COLS_KEEP].copy()
    df = df.rename(columns=RENAME)                         #Schema standardization


    # Standardise gender
    df["Gender"] = df["Gender"].map({
        "MLE":"Male","FMLE":"Female","BTSX":"Both",
        "Male":"Male","Female":"Female","Both":"Both"
    }).fillna("Both")                                       #Categorical normalization

    # Year as int, filter 2012-2022
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    df = df[df["Year"].between(2012, 2022)].reset_index(drop=True)

    
    # Convert country codes
    df["Country"] = df["Country"].apply(code_to_country)    #Element wise transformation


    # Ensure numerics
    for col in ["Mean_Estimate","LowerBound","UpperBound"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")


    # Feature engineering
    df["CI_Width"] = (df["UpperBound"] - df["LowerBound"]).round(2)
    df[label_col] = df["Mean_Estimate"].apply(level_fn)     # Classififcation/Categorization
    df = df.dropna(subset=["Mean_Estimate","Country"])
    return df

def obesity_level(v):
    if v >= 30: return "High"
    if v >= 25: return "Moderate"
    return "Low"

def malnutrition_level(v):
    if v >= 20: return "High"
    if v >= 10: return "Moderate"
    return "Low"

df_obesity     = clean_df(df_obesity,     "Obesity_Level",     obesity_level)
df_malnutrition= clean_df(df_malnutrition,"Malnutrition_Level",malnutrition_level)

# Save processed
df_obesity.to_csv("../data/processed/obesity_clean.csv", index=False)
df_malnutrition.to_csv("../data/processed/malnutrition_clean.csv", index=False)

print("Obesity shape:",     df_obesity.shape)
print("Malnutrition shape:",df_malnutrition.shape)
print(df_obesity.dtypes)

Obesity shape: (27720, 10)
Malnutrition shape: (27720, 10)
Region            object
Gender            object
Year               Int64
LowerBound       float64
UpperBound       float64
Mean_Estimate    float64
Country           object
age_group         object
CI_Width         float64
Obesity_Level     object
dtype: object
